In [ ]:
import re
import pandas as pd
import ir_datasets
from trectools import TrecEval


def cast_relevance(val):
    val = str(val)
    if "<think>" in val or "</think>" in val:
        # minimax thinking mode is output verbatim, so we need to filter it
        val = val.rsplit("</think>", 1)[-1]
    # first integer occuring (sometimes the models include markdown formatting like **1**)
    m = re.search(r'-?\d+', val)
    m = int(m.group()) if m else 0
    m = m if m in [0,1,2,3] else 0
    return m

In [ ]:
qrels = (
    (
        pd.concat([
            (
                pd.DataFrame(ir_datasets.load("msmarco-passage/trec-dl-2020/judged").qrels_iter())
                .drop("iteration", axis="columns")
                .assign(model="")
                .assign(relevance=lambda df: (df["relevance"] > 0).astype(int))
            ),
            pd.concat([
                pd.read_json("../data/processed/predictions/predictions_ministral.jsonl.xz", lines=True).assign(model="ministral"),
                pd.read_json("../data/processed/predictions/predictions_qwen-3.jsonl.xz", lines=True).assign(model="qwen3"),
                pd.read_json("../data/processed/predictions/predictions_gpt-4o.jsonl.xz", lines=True).assign(model="gpt4o"),
                pd.read_json("../data/processed/predictions/predictions_gpt-oss.jsonl.xz", lines=True).assign(model="gpt-oss"),
                pd.read_json("../data/processed/predictions/predictions_minimax.jsonl.xz", lines=True).assign(model="minimax"),
            ]).assign(relevance=lambda df: (df["relevance"].apply(cast_relevance) > 0).astype(int)),
            pd.concat([
                pd.read_parquet("../data/processed/trecdl-shallow-qrels-64.parquet").assign(model="adapter", prompt="64"),
                pd.read_parquet("../data/processed/trecdl-shallow-qrels-128.parquet").assign(model="adapter", prompt="128"),
                pd.read_parquet("../data/processed/trecdl-shallow-qrels-192.parquet").assign(model="adapter", prompt="192"),
                pd.read_parquet("../data/processed/trecdl-shallow-qrels-256.parquet").assign(model="adapter", prompt="256"),
            ]).assign(relevance=lambda df: (df["relevance"] > 0.5).astype(int)),
            # to get correct query/doc ids; rel is replaced with 0
            pd.read_parquet("../data/processed/trecdl-shallow-qrels-256.parquet").assign(model="baseline", prompt="").assign(relevance=0),
        ])
        .assign(model=lambda df: "relevance_"+df["model"].str.lower()+"_"+df["prompt"].fillna(""))
        .replace({"relevance__": "relevance", "relevance_baseline_": "relevance_baseline"})
        .astype({"query_id": str, "doc_id": str, "relevance": str})
        .assign(relevance=lambda df: df["relevance"].apply(cast_relevance))
        .reset_index()
        .drop(["index", "prompt"], axis=1)
        .drop_duplicates(subset=["query_id", "doc_id", "model"])
    )
    .pivot(index=["query_id", "doc_id"], columns="model", values="relevance")
    .assign(train=lambda df: df["relevance_adapter_256"].isna())
)


for col in qrels.columns:
   if col.startswith("relevance_"):
       qrels[col] = qrels[col].where(~qrels["train"], qrels["relevance"])

qrels = qrels.reset_index().astype({
   "query_id": str,
   "doc_id": str,
   **{c: float for c in filter(lambda x: x.startswith("relevance"), qrels.columns)},
   "train": bool
}).drop_duplicates(subset=["query_id", "doc_id"])
qrels

In [ ]:
row_order = [
    "relevance_adapter_64",
    "relevance_adapter_128",
    "relevance_adapter_192",
    "relevance_adapter_256",

    "relevance_ministral_human",
    "relevance_qwen3_human",
    "relevance_gpt-oss_human",
    "relevance_minimax_human",
    "relevance_gpt4o_human",

    "relevance_ministral_umbrela",
    "relevance_qwen3_umbrela",
    "relevance_gpt-oss_umbrela",
    "relevance_minimax_umbrela",
    "relevance_gpt4o_umbrela",

    "relevance_ministral_few_shot_human_8",
    "relevance_qwen3_few_shot_human_8",
    "relevance_gpt-oss_few_shot_human_8",
    "relevance_minimax_few_shot_human_8",
    "relevance_gpt4o_few_shot_human_8",

    "relevance_ministral_few_shot_umbrela_8",
    "relevance_qwen3_few_shot_umbrela_8",
    "relevance_gpt-oss_few_shot_umbrela_8",
    "relevance_minimax_few_shot_umbrela_8",
    "relevance_gpt4o_few_shot_umbrela_8",

    "relevance_baseline",
]

In [ ]:
import krippendorff

df_reliability = []

for c in [col for col in qrels if col.startswith("relevance")]:
    data = (
        qrels
        .query("train == False")
        .loc[:, ["relevance", c]]
        .values
        .T
    )
    alpha = krippendorff.alpha(reliability_data=data, level_of_measurement="nominal", value_domain=(0, 1))
    df_reliability.append({
        "qrels": c,
        "alpha": alpha,
    })

df_reliability = pd.DataFrame(df_reliability)
df_reliability.set_index("qrels").loc[row_order, :]

In [ ]:
import trectools
from glob import glob
from tqdm.notebook import tqdm

runs = [trectools.TrecRun(f) for f in glob(f"../data/external/dl20-passages-runs/*.gz")]
depths = [5, 10, 50, 100, 1000]
eval_data = []

for c in [col for col in qrels if col.startswith("relevance")]:
    jdg = trectools.TrecQrel()
    jdg.qrels_data = (
        qrels
        .loc[:, ["query_id", "doc_id", c]]
        .rename({"query_id": "query", "doc_id": "docid",c: "rel"}, axis="columns")
    )
    for r in tqdm(runs, desc=c):
        eval = trectools.TrecEval(r, jdg)
        eval_data.append(
            pd.concat([
                pd.concat([eval.get_reciprocal_rank(depth=k, per_query=True) for k in depths],
                    axis=1).reset_index().melt(id_vars=["query"]).rename(columns={"variable": "metric"}),
                pd.concat([eval.get_map(depth=k, per_query=True) for k in depths],
                    axis=1).reset_index().melt(id_vars=["query"]).rename(columns={"variable": "metric"}),
                pd.concat([eval.get_precision(depth=k, per_query=True) for k in depths],
                    axis=1).reset_index().melt(id_vars=["query"]).rename(columns={"variable": "metric"}),
                pd.concat([eval.get_recall(depth=k, per_query=True) for k in depths],
                    axis=1).reset_index().melt(id_vars=["query"]).rename(columns={"variable": "metric"}),
                pd.concat([eval.get_ndcg(depth=k, per_query=True) for k in depths],
                    axis=1).reset_index().melt(id_vars=["query"]).rename(columns={"variable": "metric"}),
                pd.concat([eval.get_unjudged(depth=k, per_query=True) for k in depths],
                    axis=1).reset_index().melt(id_vars=["query"]).rename(columns={"variable": "metric"})
            ])
            .reset_index()
            .assign(
                system=r.get_runid(),
                source=c
            )
        )
df = pd.concat(eval_data)

In [ ]:
df.drop("index", axis="columns").reset_index(drop=True).astype(str).to_parquet("scores.parquet")

In [ ]:
eval_topics = (
    df
    .query("source == 'relevance'")
    .drop("index", axis=1)
    .pivot(index=["system", "query"], columns="metric", values="value")
    .dropna()
    .reset_index()
    .loc[:, "query"]
    .unique()
)

In [ ]:
(
    res.loc[
        row_order,
        ["recip_rank@5", "recip_rank@10", "recip_rank@50", "NDCG@5", "NDCG@10", "NDCG@50", "MAP@5", "MAP@10", "MAP@50", "Bpref@5", "Bpref@10", "Bpref@50"],
    ]
    .transpose()
    .reset_index()
    .assign(
        depth=lambda df: df["metric"].apply(lambda row: row.split("@")[-1]).astype(int),
        metric=lambda df: df["metric"].apply(lambda row: row.split("@")[0]).astype(str),
    )
    .set_index(["metric", "depth"])
    .transpose()
)

In [ ]:
df["metric"].unique()

In [ ]:
res = (
    df
    .query("query.isin(@eval_topics)")
    .drop(["query", "index"], axis="columns")
    .dropna(subset="value")
    .astype({"value": float, "system": str, "source": str})
    .reset_index(drop=True)
    .groupby(["metric", "system", "source"])
    .mean()
    .reset_index()
    .pivot(index=["system", "metric"], columns="source", values="value")
    .reset_index()
    .set_index("system")
    .groupby("metric")
    .corr(method="spearman")
    .reset_index()
    .query("source == 'relevance'")
    .drop(["source", "relevance"], axis=1)
    .set_index("metric")
    .transpose()
    .loc[row_order, :]
    .loc[row_order, ['P@5', 'P@10', 'P@50']]
)
res

In [ ]:
def get_group(g):
    if "few_shot" in g:
        return "few_shot"
    elif "adapter" in g:
        return "adapter"
    else:
        return "zero-shot"

(
    res
    .reset_index()
    .assign(group=lambda df: df["source"].apply(lambda row: get_group(row)))
    .groupby("group")
    .apply(lambda group: group.apply(lambda col: group["source"].iloc[int(col.argmax())]))
    .transpose()
)

In [ ]:
(
    qrels
    .loc[:, ["relevance", "relevance_adapter_256"]]
    .value_counts()
    .reset_index()
    .pivot(index="relevance", columns="relevance_adapter_256", values="count")
)

In [ ]:
(
    qrels
    .loc[:, ["relevance", "relevance_gpt-oss_human"]]
    .value_counts()
    .reset_index()
    .pivot(index="relevance", columns="relevance_gpt-oss_human", values="count")
)

In [ ]:
import pandas as pd
import numpy as np
import statsmodels.stats.api as sms


df = pd.read_parquet("scores.parquet").replace({"nan": np.nan})

eval_topics = (
    df
    .query("source == 'relevance'")
    .pivot(index=["system", "query"], columns="metric", values="value")
    .dropna()
    .reset_index()
    .loc[:, "query"]
    .unique()
)

df = df.query("query in @eval_topics")

In [ ]:
import seaborn as sns
import matplotlib.pyplot as plt

metrics = [
    #'Bpref@10', 'Bpref@50',
    #'MAP@10',
    'NDCG@10', 'NDCG@50',
    'P@10', 'P@50',
    'R@10', 'R@50',
    'recip_rank@10', 'recip_rank@50'
]

data = (
    df
    .query("query.isin(@eval_topics)")
    #.drop(["query"], axis="columns")
    .dropna(subset="value")
    .astype({"value": float, "system": str, "source": str})
    .reset_index(drop=True)
    .pivot(index=["system", "metric", "query"], columns="source", values="value")
    .reset_index()
    .set_index("system")
    .groupby(["metric", "query"])
    .corr(method="spearman")
    .reset_index()
    .query("source == 'relevance'")
    .drop(["source", "relevance"], axis=1)
    .melt(id_vars=["metric", "query"], value_name="value")
    .dropna(subset="value")
)

(
    data
    .drop(columns=["query"])
    .groupby(["source", "metric"])
    .apply(lambda group: pd.Series({k: v[0] for k,v in zip(["upper", "lower"], sms.DescrStatsW(group).tconfint_mean())} | {"mean": group["value"].mean()}))
    .reset_index()
    .query("metric in @metrics")
)


In [ ]:
from scipy.stats import ttest_rel

test_res = []
for metric in metrics:
    for adapter in ['relevance_adapter_64', 'relevance_adapter_128', 'relevance_adapter_192', 'relevance_adapter_256']:
        for llm in ['relevance_qwen3_human', "relevance_gpt4o_umbrela", "relevance_qwen3_few_shot_human_8", "relevance_qwen3_few_shot_umbrela_8"]:
            for metric in ["NDCG@10", "NDCG@50"]:
                a = data.sort_values("query").loc[
                    (data["source"] == adapter) &
                    (data["metric"] == metric),
                    "value"
                ]
                b = data.sort_values("query").loc[
                    (data["source"] == llm) &
                    (data["metric"] == metric),
                    "value"
                ]
                _, p = ttest_rel(a, b, alternative="greater")
                test_res.append({
                    "adapter": adapter,
                    "llm": llm,
                    "metric": metric,
                    "p": p
                })